In [6]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_1_day_-1.csv',
    'prices_round_1_day_-2.csv',
    'prices_round_1_day_0.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [7]:
def calculate_book_vwap(df):
    nominal = 0
    volume = 0
    for i in range(1, 4):
        nominal += (df[f'bid_price_{i}'] * df[f'bid_volume_{i}']).fillna(0)
        nominal += (df[f'ask_price_{i}'] * df[f'ask_volume_{i}']).fillna(0)
        volume += df[f'bid_volume_{i}'].fillna(0)
        volume += df[f'ask_volume_{i}'].fillna(0)
    return nominal / volume.replace(0, np.nan)


def calculate_micro_price(df):
    """Volume-weighted L1 microprice using bid/ask volumes."""
    bid_p, bid_v = df['bid_price_1'], df['bid_volume_1']
    ask_p, ask_v = df['ask_price_1'], df['ask_volume_1']
    micro_price = (bid_p * ask_v + ask_p * bid_v) / (bid_v + ask_v)
    return micro_price.fillna((bid_p + ask_p) / 2)


In [8]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go

# Configuration
days = [-1, -2, 0]
window_size = 300 # Ticks for rolling average/median

for day in days:
    # Filter and copy subset
    subset = df_total[(df_total['product'] == "ASH_COATED_OSMIUM") & (df_total['day'] == day)].copy()
    
    if subset.empty:
        print(f"Skipping day {day}: No data found.")
        continue
    
    # 1. Calculate Micro Price and Handle Mid Price
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)
    subset['micro_price'] = calculate_micro_price(subset)
    
    # 2. Calculate Rolling Stats
    subset['mid_price_roll_median'] = subset['mid_price'].rolling(window=window_size).median()
    subset['micro_roll_median'] = subset['micro_price'].rolling(window=window_size).median()
    
    # 3. Statistical Analysis Suite (Residuals)
    # Using Mid Price vs Median as the primary noise baseline
    subset['mid_residuals'] = subset['mid_price'] - subset['mid_price_roll_median']
    res_clean = subset['mid_residuals'].dropna()

    if not res_clean.empty:
        # Ljung-Box for Autocorrelation (White Noise Test)
        lb_results = acorr_ljungbox(res_clean, lags=[10], return_df=True)
        lb_pvalue = lb_results['lb_pvalue'].iloc[0]
        
        # Distribution Metrics
        kurt = stats.kurtosis(res_clean)
        skew = stats.skew(res_clean)
        _, norm_p = stats.normaltest(res_clean)

        print(f"\n--- Statistical Suite: Day {day} ---")
        print(f"Ljung-Box (Lag 10) p-value: {lb_pvalue:.5f}")
        print(f"  > Status: {'White Noise' if lb_pvalue > 0.05 else 'Signal Leakage (Autocorrelated)'}")
        print(f"Excess Kurtosis: {kurt:.2f}")
        print(f"  > Distribution: {'Fat-Tailed (Outlier Heavy)' if kurt > 0 else 'Thin-Tailed'}")
        print(f"Normality p-value: {norm_p:.5f}")
        print("-" * 35)

        # Optional: Distribution Plot
        fig_hist = go.Figure(data=[go.Histogram(x=res_clean, nbinsx=50, marker_color='gray')])
        fig_hist.update_layout(title=f"Residual Distribution - Day {day}", template='plotly_white')
        fig_hist.show()

    # 4. Create the Interactive Plotly Figure
    fig = go.Figure()


    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['mid_price_roll_median'], 
                             name='Mid Rolling Median', line=dict(color='darkblue', dash='dot')))
    
    # Order Book Levels
    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['bid_price_1'], 
                             name='Bid 1', line=dict(color='green', dash='dot', width=1)))
    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['ask_price_1'], 
                             name='Ask 1', line=dict(color='red', dash='dot', width=1)))


    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['micro_roll_median'], 
                             name='micro_price Rolling Median', line=dict(color='firebrick', dash='dot')))

    # Layout Customization
    fig.update_layout(
        title=f'Market Analysis: ASH_COATED_OSMIUM - Day {day}',
        xaxis_title='Timestamp',
        yaxis_title='Price',
        legend_title='Metrics',
        template='plotly_white',
        hovermode='x unified'
    )
    
    fig.show()


--- Statistical Suite: Day -1 ---
Ljung-Box (Lag 10) p-value: 0.00000
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 1.36
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------



--- Statistical Suite: Day -2 ---
Ljung-Box (Lag 10) p-value: 0.00000
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 0.98
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------



--- Statistical Suite: Day 0 ---
Ljung-Box (Lag 10) p-value: 0.00000
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 0.96
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------


In [9]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go

# Configuration
days = [-1, -2, 0]
window_size = 3 

for day in days:
    subset = df_total[(df_total['product'] == "ASH_COATED_OSMIUM") & (df_total['day'] == day)].copy()
    if subset.empty: continue
    
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)
    subset['mid_price_roll_median'] = subset['mid_price'].rolling(window=window_size).median()
    subset['mid_residuals'] = subset['mid_price'] - subset['mid_price_roll_median']
    res_clean = subset['mid_residuals'].dropna()

    if not res_clean.empty:
        # Fit Laplace parameters
        loc_lap, scale_lap = stats.laplace.fit(res_clean)
        
        # Create a range for the theoretical PDF curve
        x_pdf = np.linspace(res_clean.min(), res_clean.max(), 1000)
        y_pdf = stats.laplace.pdf(x_pdf, loc_lap, scale_lap)

        # Calculate metrics
        kurt = stats.kurtosis(res_clean)
        ks_stat, laplace_p = stats.kstest(res_clean, 'laplace', args=(loc_lap, scale_lap))

        print(f"\n--- Analysis: Day {day} ---")
        print(f"Excess Kurtosis: {kurt:.2f} (Target for Laplace is 3.0)")
        print(f"Laplace KS p-value: {laplace_p:.5f}")

        # Distribution Plot with PDF Overlay
        fig_hist = go.Figure()

        # Actual Residuals Histogram (Normalized to show density)
        fig_hist.add_trace(go.Histogram(
            x=res_clean, 
            histnorm='probability density', 
            name='Actual Residuals',
            marker_color='rgba(100, 100, 100, 0.6)'
        ))

        # Theoretical Laplace PDF
        fig_hist.add_trace(go.Scatter(
            x=x_pdf, 
            y=y_pdf, 
            mode='lines', 
            name='Theoretical Laplace',
            line=dict(color='red', width=2)
        ))

        fig_hist.update_layout(
            title=f"Residual Density vs Laplace Fit - Day {day} (Kurtosis: {kurt:.2f})",
            xaxis_title="Residual Value (Ticks)",
            yaxis_title="Density",
            template='plotly_white'
        )
        fig_hist.show()


--- Analysis: Day -1 ---
Excess Kurtosis: 6.98 (Target for Laplace is 3.0)
Laplace KS p-value: 0.00000



--- Analysis: Day -2 ---
Excess Kurtosis: 6.72 (Target for Laplace is 3.0)
Laplace KS p-value: 0.00000



--- Analysis: Day 0 ---
Excess Kurtosis: 7.10 (Target for Laplace is 3.0)
Laplace KS p-value: 0.00000


In [10]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go

# ----------------------------
# Causal Hampel utilities
# ----------------------------
def causal_hampel_filter(series, window_size=17, n_sigma=3.0):
    """
    Causal Hampel filter (NO look-ahead): uses only data up to current tick.
    """
    x = series.astype(float).copy()
    rolling_median = x.rolling(window=window_size, min_periods=window_size, center=False).median()

    rolling_mad = x.rolling(window=window_size, min_periods=window_size, center=False).apply(
        lambda arr: np.median(np.abs(arr - np.median(arr))), raw=True
    )
    sigma = 1.4826 * rolling_mad
    threshold = n_sigma * sigma

    residual_raw = x - rolling_median
    outlier_mask = residual_raw.abs() > threshold

    fv = x.copy()
    fv[outlier_mask] = rolling_median[outlier_mask]

    out = pd.DataFrame({
        "fair_value": fv,
        "rolling_median": rolling_median,
        "rolling_mad": rolling_mad,
        "rolling_sigma": sigma,
        "adaptive_threshold": threshold,
        "residual": x - fv,
        "is_storm": outlier_mask,
    })
    return out


def compute_signals(subset, window_size=17, n_sigma=3.0):
    """Build causal fair value, adaptive threshold, and entry signals."""
    out = subset.copy()
    out["mid_price"] = pd.to_numeric(out["mid_price"], errors="coerce").replace(0, np.nan)

    ham = causal_hampel_filter(out["mid_price"], window_size=window_size, n_sigma=n_sigma)
    out = pd.concat([out.reset_index(drop=True), ham.reset_index(drop=True)], axis=1)

    eps = out["residual"].abs()
    contraction_1 = eps < eps.shift(1)
    contraction_2 = eps.shift(1) < eps.shift(2)

    extreme = eps >= out["adaptive_threshold"]  # adaptive threshold via rolling MAD
    out["entry_wait_and_fade"] = extreme & contraction_1
    out["entry_wait_and_fade_persist2"] = extreme & contraction_1 & contraction_2

    return out


days = [-1, -2, 0]
WINDOW_SIZE = 17
N_SIGMA = 3.0
SPREAD_TAX_TICKS = 2.0

ash_by_day = {}
for day in days:
    subset = df_total[(df_total["product"] == "ASH_COATED_OSMIUM") & (df_total["day"] == day)].copy()
    if subset.empty:
        continue
    subset = subset.sort_values("timestamp").reset_index(drop=True)
    ash_by_day[day] = compute_signals(subset, window_size=WINDOW_SIZE, n_sigma=N_SIGMA)

print("Prepared causal Hampel datasets:", {k: len(v) for k, v in ash_by_day.items()})
print("Causal mode active: center=False (no future leakage).")


Prepared causal Hampel datasets: {-1: 10000, -2: 10000, 0: 10000}
Causal mode active: center=False (no future leakage).


In [11]:
import numpy as np
import scipy.stats as stats
from statsmodels.sandbox.stats.runs import runstest_1samp
from statsmodels.stats.diagnostic import acorr_ljungbox


def run_diagnostic_suite(proc_df):
    res_clean = proc_df["residual"].dropna()
    storm_mask = proc_df["is_storm"].fillna(False)

    out = {}
    if len(res_clean) < 30:
        return out

    # Student-t fit
    df_t, loc_t, scale_t = stats.t.fit(res_clean)
    _, p_t = stats.kstest(res_clean, "t", args=(df_t, loc_t, scale_t))
    out["t_fit"] = {"nu": df_t, "p_value": p_t}

    # Runs test on storm indicator
    _, p_runs = runstest_1samp(storm_mask.astype(int), cutoff="mean")
    out["runs_test_p"] = p_runs

    # Variance ratio on residual increments
    dx = res_clean.diff().dropna()
    k = 5
    if len(dx) > k + 5:
        var_1 = np.var(dx)
        var_k = np.var(dx.rolling(window=k).sum().dropna())
        out["variance_ratio"] = var_k / (k * var_1 + 1e-12)
    else:
        out["variance_ratio"] = np.nan

    # Ljung-Box on causal residuals
    lb_df = acorr_ljungbox(res_clean, lags=[10], return_df=True)
    out["ljung_box_p10"] = float(lb_df["lb_pvalue"].iloc[0])
    return out


for day in days:
    if day not in ash_by_day:
        continue
    proc = ash_by_day[day]
    diag = run_diagnostic_suite(proc)
    if not diag:
        print(f"Day {day}: insufficient data")
        continue

    status = "white-ish" if diag["ljung_box_p10"] > 0.05 else "autocorrelated"
    print(f"\n==== Causal Diagnostics: Day {day} ====")
    print(f"Ljung-Box p(10): {diag['ljung_box_p10']:.5f} ({status})")
    print(f"Student-t nu: {diag['t_fit']['nu']:.3f}, KS p: {diag['t_fit']['p_value']:.5f}")
    print(f"Runs test p: {diag['runs_test_p']:.5f}")
    print(f"Variance ratio (k=5 on dResidual): {diag['variance_ratio']:.4f}")
    print(f"Storm rate: {proc['is_storm'].mean() * 100:.2f}%")
    print(f"Median adaptive threshold: {proc['adaptive_threshold'].median():.3f} ticks")



==== Causal Diagnostics: Day -1 ====
Ljung-Box p(10): 0.57341 (white-ish)
Student-t nu: 1.988, KS p: 0.00000
Runs test p: 0.00000
Variance ratio (k=5 on dResidual): 0.2038
Storm rate: 18.68%
Median adaptive threshold: 4.448 ticks

==== Causal Diagnostics: Day -2 ====
Ljung-Box p(10): 0.45421 (white-ish)
Student-t nu: 1.715, KS p: 0.00000
Runs test p: 0.00000
Variance ratio (k=5 on dResidual): 0.2019
Storm rate: 18.40%
Median adaptive threshold: 4.448 ticks

==== Causal Diagnostics: Day 0 ====
Ljung-Box p(10): 0.70795 (white-ish)
Student-t nu: 1.715, KS p: 0.00000
Runs test p: 0.00000
Variance ratio (k=5 on dResidual): 0.2035
Storm rate: 18.00%
Median adaptive threshold: 4.448 ticks


In [12]:
def test_signal_edge(proc_df, horizons=(3, 5, 10), use_persistence=True):
    entry_col = "entry_wait_and_fade_persist2" if use_persistence else "entry_wait_and_fade"
    rows = []
    sig = proc_df[entry_col].fillna(False)
    signed_res = -np.sign(proc_df["residual"])

    for h in horizons:
        fwd = proc_df["mid_price"].shift(-h) - proc_df["mid_price"]
        signed = fwd * signed_res
        edge_raw = signed[sig].mean()
        edge_net = edge_raw - SPREAD_TAX_TICKS
        rows.append({
            "horizon": h,
            "trades": int(sig.sum()),
            "edge_raw": edge_raw,
            "edge_net_spread": edge_net,
        })
    return pd.DataFrame(rows)


for day in days:
    if day not in ash_by_day:
        continue
    proc = ash_by_day[day]
    print(f"\n--- Day {day}: Edge table (causal) ---")
    display(test_signal_edge(proc, horizons=(3, 5, 10, 15), use_persistence=True))



--- Day -1: Edge table (causal) ---


,horizon,trades,edge_raw,edge_net_spread
0,3,60,0.325000,-1.675000
1,5,60,0.058333,-1.941667
2,10,60,0.135593,-1.864407
3,15,60,0.033333,-1.966667



--- Day -2: Edge table (causal) ---


,horizon,trades,edge_raw,edge_net_spread
0,3,56,0.526786,-1.473214
1,5,56,0.276786,-1.723214
2,10,56,0.482143,-1.517857
3,15,56,0.330357,-1.669643



--- Day 0: Edge table (causal) ---


,horizon,trades,edge_raw,edge_net_spread
0,3,46,-0.184783,-2.184783
1,5,46,0.717391,-1.282609
2,10,46,0.076087,-1.923913
3,15,46,0.967391,-1.032609


In [13]:
def test_alpha_durability(proc_df, horizons=(1, 3, 5, 10, 20), use_persistence=True):
    entry_col = "entry_wait_and_fade_persist2" if use_persistence else "entry_wait_and_fade"
    sig = proc_df[entry_col].fillna(False)

    print(f"{'Horizon':<10} | {'IC (Spearman)':<15} | {'Edge Raw':<10} | {'Edge Net':<10} | {'Trades':<8}")
    print("-" * 78)

    for h in horizons:
        fwd = proc_df["mid_price"].shift(-h) - proc_df["mid_price"]
        valid = ~(proc_df["residual"].isna() | fwd.isna())
        ic = stats.spearmanr(proc_df.loc[valid, "residual"], fwd[valid])[0] if valid.sum() > 10 else np.nan

        signed = fwd * -np.sign(proc_df["residual"])
        edge_raw = signed[sig].mean()
        edge_net = edge_raw - SPREAD_TAX_TICKS

        print(f"{h:<10} | {ic:<15.4f} | {edge_raw:<10.4f} | {edge_net:<10.4f} | {int(sig.sum()):<8}")


for day in days:
    if day not in ash_by_day:
        continue
    print(f"\n=== Day {day}: Alpha durability (causal + persistence) ===")
    test_alpha_durability(ash_by_day[day], horizons=(1, 3, 5, 10, 20), use_persistence=True)



=== Day -1: Alpha durability (causal + persistence) ===
Horizon    | IC (Spearman)   | Edge Raw   | Edge Net   | Trades  
------------------------------------------------------------------------------
1          | -0.4564         | 0.2167     | -1.7833    | 60      
3          | -0.4459         | 0.3250     | -1.6750    | 60      
5          | -0.4427         | 0.0583     | -1.9417    | 60      
10         | -0.4242         | 0.1356     | -1.8644    | 60      
20         | -0.4153         | 0.0333     | -1.9667    | 60      

=== Day -2: Alpha durability (causal + persistence) ===
Horizon    | IC (Spearman)   | Edge Raw   | Edge Net   | Trades  
------------------------------------------------------------------------------
1          | -0.4576         | 0.2500     | -1.7500    | 56      
3          | -0.4557         | 0.5268     | -1.4732    | 56      
5          | -0.4450         | 0.2768     | -1.7232    | 56      
10         | -0.4431         | 0.4821     | -1.5179    | 56      
20

In [14]:
def test_trade_capacity(proc_df, horizons=(10,), scale_map=None):
    """Capacity proxy with spread tax + residual-proportional sizing."""
    if scale_map is None:
        scale_map = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}

    sig = proc_df["entry_wait_and_fade_persist2"].fillna(False)
    abs_res = proc_df["residual"].abs()
    bucket = abs_res.round().clip(lower=1, upper=max(scale_map))
    lots = bucket.map(scale_map).fillna(1)

    for h in horizons:
        fwd = proc_df["mid_price"].shift(-h) - proc_df["mid_price"]
        signed = fwd * -np.sign(proc_df["residual"])

        pnl_raw = signed[sig]
        lots_sig = lots[sig]
        gross_ticks = (pnl_raw * lots_sig).sum()
        spread_cost = SPREAD_TAX_TICKS * lots_sig.sum()
        net_ticks = gross_ticks - spread_cost

        n_trades = int(sig.sum())
        avg_lots = lots_sig.mean() if n_trades > 0 else 0.0
        avg_edge_raw = pnl_raw.mean() if n_trades > 0 else np.nan
        avg_edge_net = (pnl_raw - SPREAD_TAX_TICKS).mean() if n_trades > 0 else np.nan

        print(f"H={h} | trades={n_trades} | avg_lots={avg_lots:.2f}")
        print(f"    gross_ticks={gross_ticks:.2f} | spread_cost={spread_cost:.2f} | net_ticks={net_ticks:.2f}")
        print(f"    avg_edge_raw={avg_edge_raw:.3f} | avg_edge_net={avg_edge_net:.3f}")


for day in days:
    if day not in ash_by_day:
        continue
    print(f"\n=== Day {day}: Capacity check (spread-adjusted) ===")
    test_trade_capacity(ash_by_day[day], horizons=(5, 10, 15))



=== Day -1: Capacity check (spread-adjusted) ===
H=5 | trades=60 | avg_lots=1.07
    gross_ticks=39.50 | spread_cost=128.00 | net_ticks=-88.50
    avg_edge_raw=0.058 | avg_edge_net=-1.942
H=10 | trades=60 | avg_lots=1.07
    gross_ticks=42.00 | spread_cost=128.00 | net_ticks=-86.00
    avg_edge_raw=0.136 | avg_edge_net=-1.864
H=15 | trades=60 | avg_lots=1.07
    gross_ticks=6.00 | spread_cost=128.00 | net_ticks=-122.00
    avg_edge_raw=0.033 | avg_edge_net=-1.967

=== Day -2: Capacity check (spread-adjusted) ===
H=5 | trades=56 | avg_lots=1.05
    gross_ticks=19.50 | spread_cost=118.00 | net_ticks=-98.50
    avg_edge_raw=0.277 | avg_edge_net=-1.723
H=10 | trades=56 | avg_lots=1.05
    gross_ticks=33.50 | spread_cost=118.00 | net_ticks=-84.50
    avg_edge_raw=0.482 | avg_edge_net=-1.518
H=15 | trades=56 | avg_lots=1.05
    gross_ticks=25.50 | spread_cost=118.00 | net_ticks=-92.50
    avg_edge_raw=0.330 | avg_edge_net=-1.670

=== Day 0: Capacity check (spread-adjusted) ===
H=5 | trades=

In [15]:
# Grid-search W on training days (-1, 0), evaluate best on holdout day (-2)
WINDOW_GRID = [9, 11, 13, 15, 17, 21]
TRAIN_DAYS = [-1, 0]
HOLDOUT_DAY = -2


def evaluate_window(day_df, w, n_sigma=3.0, horizon=10):
    tmp = day_df.copy()
    tmp["mid_price"] = pd.to_numeric(tmp["mid_price"], errors="coerce").replace(0, np.nan)
    ham = causal_hampel_filter(tmp["mid_price"], window_size=w, n_sigma=n_sigma)
    tmp = pd.concat([tmp.reset_index(drop=True), ham.reset_index(drop=True)], axis=1)

    eps = tmp["residual"].abs()
    c1 = eps < eps.shift(1)
    c2 = eps.shift(1) < eps.shift(2)
    extreme = eps >= tmp["adaptive_threshold"]
    sig = (extreme & c1 & c2).fillna(False)

    fwd = tmp["mid_price"].shift(-horizon) - tmp["mid_price"]
    signed = fwd * -np.sign(tmp["residual"])
    edge_raw = signed[sig].mean()
    edge_net = edge_raw - SPREAD_TAX_TICKS

    valid = ~(tmp["residual"].isna() | fwd.isna())
    ic = stats.spearmanr(tmp.loc[valid, "residual"], fwd[valid])[0] if valid.sum() > 10 else np.nan
    lb_p = acorr_ljungbox(tmp["residual"].dropna(), lags=[10], return_df=True)["lb_pvalue"].iloc[0]

    return {
        "W": w,
        "trades": int(sig.sum()),
        "edge_raw_h10": edge_raw,
        "edge_net_h10": edge_net,
        "IC_h10": ic,
        "LB_p10": float(lb_p),
        "median_threshold": float(tmp["adaptive_threshold"].median()) if tmp["adaptive_threshold"].notna().any() else np.nan,
    }


day_raw = {}
for d in days:
    sub = df_total[(df_total["product"] == "ASH_COATED_OSMIUM") & (df_total["day"] == d)].copy()
    day_raw[d] = sub.sort_values("timestamp").reset_index(drop=True)

rows_train = []
for w in WINDOW_GRID:
    metrics = []
    for d in TRAIN_DAYS:
        m = evaluate_window(day_raw[d], w, n_sigma=3.0, horizon=10)
        m["day"] = d
        metrics.append(m)
    dfm = pd.DataFrame(metrics)
    rows_train.append({
        "W": w,
        "train_edge_net_mean": dfm["edge_net_h10"].mean(),
        "train_IC_mean": dfm["IC_h10"].mean(),
        "train_LB_p10_mean": dfm["LB_p10"].mean(),
        "train_trades_mean": dfm["trades"].mean(),
    })

train_table = pd.DataFrame(rows_train).sort_values("train_edge_net_mean", ascending=False)
print("\n=== TRAIN (-1,0) WINDOW GRID ===")
display(train_table)

best_w = int(train_table.iloc[0]["W"])
print(f"Best W by net edge on train: {best_w}")

holdout_metrics = evaluate_window(day_raw[HOLDOUT_DAY], best_w, n_sigma=3.0, horizon=10)
print("\n=== HOLDOUT DAY (-2) METRICS @ best W ===")
print(holdout_metrics)



=== TRAIN (-1,0) WINDOW GRID ===


,W,train_edge_net_mean,train_IC_mean,train_LB_p10_mean,train_trades_mean
5,21,-1.747409,-0.434860,0.583137,40.5
0,9,-1.777698,-0.429676,0.693136,60.0
2,13,-1.839120,-0.436925,0.705511,63.5
4,17,-1.894160,-0.435043,0.640682,53.0
3,15,-1.926471,-0.435154,0.683019,60.0
1,11,-1.949301,-0.433984,0.737753,69.0


Best W by net edge on train: 21

=== HOLDOUT DAY (-2) METRICS @ best W ===
{'W': 21, 'trades': 50, 'edge_raw_h10': np.float64(0.31), 'edge_net_h10': np.float64(-1.69), 'IC_h10': np.float64(-0.4414545690022191), 'LB_p10': 0.47801771397237713, 'median_threshold': 4.4478}
